# Group Relative Policy Optimization (GRPO) with veRL on Amazon SageMaker Training jobs
## Lab 3 - LLM Deployment
In this notebook, we are going to deploy the GRPO-trained model to a SageMaker real-time endpoint using an inference component and the AWS vLLM container.

## Prerequisites

### Install requirements

In [ ]:
%pip install -r ./requirements.txt --upgrade

### Setup and dependencies

In [ ]:
import json
import os

import boto3
from rich.pretty import pprint
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()

sagemaker_session_bucket = None
if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sm_client = boto3.client("sagemaker")
sagemaker_runtime = boto3.client("sagemaker-runtime")

sess = Session(default_bucket=sagemaker_session_bucket)

bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix
region = sess.boto_region_name

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {bucket_name}")
print(f"sagemaker session region: {region}")

***

## Locate the trained model
Lab 2 submitted the job with `wait=False` and did not hand anything to this notebook, so we find the trained model the same way the other options in this solution do: search for the most recent *completed* job whose name starts with the prefix we used. That keeps the labs independent -- this one works in a fresh kernel, and re-running Lab 2 simply means this cell picks up the newer run.

Utility function to find the most recent completed job for a prefix.

In [ ]:
def get_last_job_name(job_name_prefix):
    search_params = {
        "Resource": "TrainingJob",
        "SearchExpression": {
            "Filters": [
                {
                    "Name": "TrainingJobName",
                    "Operator": "Contains",
                    "Value": job_name_prefix,
                },
                {
                    "Name": "TrainingJobStatus",
                    "Operator": "Equals",
                    "Value": "Completed",
                },
            ]
        },
        "SortBy": "CreationTime",
        "SortOrder": "Descending",
        "MaxResults": 1,
    }

    results = sm_client.search(**search_params)["Results"]
    if not results:
        raise ValueError(
            f"No completed training jobs found starting with prefix '{job_name_prefix}'. "
            "If Lab 2 is still running, wait for it to reach Completed."
        )
    return results[0]["TrainingJob"]["TrainingJobName"]

In [ ]:
job_prefix = "train-qwen3-4b-grpo"

job_name = get_last_job_name(job_prefix)

if default_prefix:
    output_path = f"{default_prefix}/grpo-verl"
else:
    output_path = "grpo-verl"

# The exported Hugging Face model. Lab 2 set compression_type="NONE", so this is a
# browsable prefix rather than a model.tar.gz, which is what the serving container
# mounts directly.
model_s3_uri = f"s3://{bucket_name}/{output_path}/{job_name}/output/model/"

print(f"job:   {job_name}")
print(f"model: {model_s3_uri}")

What sits behind that prefix is worth a moment, because it is not what veRL wrote. veRL checkpoints the actor as FSDP shards, which nothing outside veRL can load. `scripts/export_checkpoint.py` runs at the end of training, merges those shards, and writes a plain Hugging Face model directory -- config, safetensors, tokenizer. That is why the serving container can be an ordinary vLLM container that knows nothing about veRL: what crosses between training and serving is a standard model directory, not an RL framework artifact.

Let's confirm the export landed.

In [ ]:
listing = s3_client.list_objects_v2(
    Bucket=bucket_name, Prefix=model_s3_uri.replace(f"s3://{bucket_name}/", "")
)

for obj in listing.get("Contents", []):
    print(f"  {obj['Size'] / 1e9:>6.2f} GB  {obj['Key'].split('/')[-1]}")

if not listing.get("Contents"):
    raise ValueError(f"nothing found under {model_s3_uri}")

### Fix tokenizer metadata before deployment
Qwen exports can store `extra_special_tokens` as a list, but the Transformers version in the vLLM inference container expects a mapping and otherwise fails with `'list' object has no attribute 'keys'`. The failure happens inside the container during model load, so from the outside all you see is an inference component that never reaches `InService`.

The next cell moves those tokens to `additional_special_tokens` in `tokenizer_config.json`. It rewrites the same S3 model path we deploy below, and keeps a backup of the original outside the model prefix so the deployed prefix stays clean.

In [ ]:
from urllib.parse import urlparse


def ensure_vllm_tokenizer_compatibility(model_s3_uri, s3_client=None):
    parsed_uri = urlparse(model_s3_uri)
    if parsed_uri.scheme != "s3" or not parsed_uri.netloc:
        raise ValueError("model_s3_uri must be an S3 URI")

    model_prefix = parsed_uri.path.strip("/")
    if not model_prefix:
        raise ValueError("model_s3_uri must include a non-empty model prefix")

    client = s3_client or boto3.client("s3")
    bucket = parsed_uri.netloc
    config_key = f"{model_prefix}/tokenizer_config.json"

    response = client.get_object(Bucket=bucket, Key=config_key)
    original_body = response["Body"].read()
    tokenizer_config = json.loads(original_body)

    extra_tokens = tokenizer_config.get("extra_special_tokens")
    if extra_tokens is None or isinstance(extra_tokens, dict):
        return {"updated": False, "config_key": config_key}

    if not isinstance(extra_tokens, list):
        raise TypeError(
            "extra_special_tokens must be a mapping or list, received "
            f"{type(extra_tokens).__name__}"
        )

    existing = tokenizer_config.get("additional_special_tokens")
    if existing is not None and existing != extra_tokens:
        raise ValueError("additional_special_tokens conflicts with extra_special_tokens")

    tokenizer_config["additional_special_tokens"] = extra_tokens
    del tokenizer_config["extra_special_tokens"]

    # Back the original up outside the model prefix, so the deployed prefix contains
    # only what the container should load.
    backup_key = f"{model_prefix.rstrip('/')}-tokenizer-backup/tokenizer_config.json"
    client.put_object(Bucket=bucket, Key=backup_key, Body=original_body)

    client.put_object(
        Bucket=bucket,
        Key=config_key,
        Body=json.dumps(tokenizer_config, indent=2).encode("utf-8"),
        ContentType="application/json",
    )

    return {"updated": True, "config_key": config_key, "backup_key": backup_key}


pprint(ensure_vllm_tokenizer_compatibility(model_s3_uri, s3_client=s3_client))

## Serving configuration
Names first, derived from the model id so they are the same every time this notebook runs.

In [ ]:
model_id = "Qwen/Qwen3-4B"

model_name = f"{model_id.split('/')[-1].replace('.', '-')}-grpo"
endpoint_config_name = model_name
endpoint_name = f"{model_name}-endpoint"
ic_name = f"{model_name}-ic"

print(f"model:           {model_name}")
print(f"endpoint config: {endpoint_config_name}")
print(f"endpoint:        {endpoint_name}")
print(f"component:       {ic_name}")

### The serving container
Serving does not need the veRL container. Training ran an actor, a reference policy, an optimizer, and a rollout engine at once; serving runs a rollout engine and nothing else. So this is the AWS-published vLLM Deep Learning Container, and the model it loads is the ordinary Hugging Face directory the export produced.

The image is addressed by digest rather than tag. A digest is content-addressed, so it names the same image in every region, and unlike a tag it cannot be moved after the fact. The tag is recorded next to it for traceability only.

The AMI version is not optional and has to be matched to the container, not merely set to "a GPU AMI". This container is built on CUDA 13, which needs NVIDIA driver 580, and `al2023-ami-sagemaker-inference-gpu-4-1` is the only `InferenceAmiVersion` that provides it. Omit it and the default host AMI's driver is too old, which surfaces as a container that dies at start with a message that never mentions the driver.

In [ ]:
# AWS Deep Learning Container registry. This account id is AWS's own and is the same
# for every customer.
CONTAINER_TAG = "server-sagemaker-cuda-v2.2.2"
CONTAINER_DIGEST = "sha256:71714c6e41b883caf4878efd41a0ab9d6e9fb63e8ac0b4aba92c6db5c9a70eca"

image_uri = f"763104351884.dkr.ecr.{region}.amazonaws.com/vllm@{CONTAINER_DIGEST}"

INFERENCE_AMI_VERSION = "al2023-ami-sagemaker-inference-gpu-4-1"

print(f"image: {image_uri}")
print(f"tag:   {CONTAINER_TAG}")

### Endpoint configuration, with a capacity fallback
One L40S is ample for serving a 4B model: the weights are about 8 GB in bf16 and the rest of the 48 GB card becomes KV cache, which is what actually determines throughput.

The list of instance types needs explaining, because a single type would be the obvious thing to write and it is what we tried first. `ml.g6e.2xlarge` endpoints failed twice with `InsufficientInstanceCapacity`, days apart, taking 38 minutes each time to report it. That error is not a defect in the request -- the quota was available and the configuration was valid, the region simply had no instance of that size to give. The endpoint is not attached to a VPC, so SageMaker was free to place it in any zone and still found nothing.

`InstancePools` is the remedy SageMaker's own error message recommends: list several instance types in priority order and the endpoint walks down the list instead of failing. Each size draws on a separate capacity pool, so a second candidate is a genuinely different chance rather than a retry. All three below carry a single L40S, so the model and the component's requirements are unchanged by which one wins. Falling back costs more per hour -- and a more expensive endpoint that exists beats a cheaper one that does not.

`variant_instance_provision_timeout_in_seconds` is what makes the list worth having. Without it the service default applies to every pool in turn, and three exhausted pools would take longer to fail than the single type this replaced.

In [ ]:
from sagemaker.core.resources import EndpointConfig
from sagemaker.core.shapes import InstancePool, ProductionVariant

# Tried in priority order. All single-L40S, so the component's requirements hold for
# every candidate.
instance_pools = [
    InstancePool(instance_type="ml.g6e.2xlarge", priority=1),
    InstancePool(instance_type="ml.g6e.4xlarge", priority=2),
    InstancePool(instance_type="ml.g6e.8xlarge", priority=3),
]

print(f"Creating EndpointConfig: {endpoint_config_name}")

endpoint_config = EndpointConfig.create(
    endpoint_config_name=endpoint_config_name,
    execution_role_arn=role,
    production_variants=[
        ProductionVariant(
            variant_name="AllTraffic",
            initial_instance_count=1,
            # Mutually exclusive with instance_type: a variant names one type or a
            # priority-ordered list of them, never both.
            instance_pools=instance_pools,
            variant_instance_provision_timeout_in_seconds=600,
            model_data_download_timeout_in_seconds=1200,
            inference_ami_version=INFERENCE_AMI_VERSION,
            routing_config={"routing_strategy": "LEAST_OUTSTANDING_REQUESTS"},
        )
    ],
)

pprint(endpoint_config.endpoint_config_name)

### Create Endpoint
A SageMaker Endpoint is a fully managed, always-on HTTPS API that hosts your deployed model and serves real-time inference requests.

**The endpoint bills from the moment it reaches `InService`, whether or not it is invoked, and it keeps billing even if the inference component below fails.** Unlike a training job it does not stop on its own. The teardown cells at the end of this notebook are not optional.

In [ ]:
from sagemaker.core.resources import Endpoint

print(f"Creating Endpoint: {endpoint_name}")

endpoint = Endpoint.create(
    endpoint_name=endpoint_name, endpoint_config_name=endpoint_config_name
)
endpoint.wait_for_status("InService")

print(f"Endpoint {endpoint_name} is InService")

### Create Model
The environment variables are how the vLLM container is configured. Two of them track decisions made back in Lab 2 rather than being tuned here.

`SM_VLLM_MAX_MODEL_LEN` is 2048 because that is exactly the budget the model was trained with -- `max_prompt_length` 1024 plus `max_response_length` 1024. Qwen3-4B supports far more context, but serving beyond what was trained buys nothing for GSM8K and costs KV cache, so this tracks the training configuration rather than the model's ceiling.

`SM_VLLM_TENSOR_PARALLEL_SIZE` is 1 because serving uses one GPU. In Lab 2 the rollout engine was sharded across two, but that was to fit alongside the actor and reference policy, which are not here.

In [ ]:
from sagemaker.core.resources import Model
from sagemaker.core.shapes import (
    ContainerDefinition,
    ModelDataSource,
    S3ModelDataSource,
)

env = {
    # Where SageMaker mounts the uncompressed model prefix.
    "SM_VLLM_MODEL": "/opt/ml/model",
    "SM_VLLM_DTYPE": "bfloat16",
    # Matches the trained budget exactly: 1024 prompt + 1024 response from Lab 2.
    "SM_VLLM_MAX_MODEL_LEN": "2048",
    "SM_VLLM_MAX_NUM_SEQS": "16",
    # Leaves headroom for weights and activations on a 48 GB card.
    "SM_VLLM_GPU_MEMORY_UTILIZATION": "0.85",
    # Keeps a long prompt from blocking the scheduler behind a single prefill.
    "SM_VLLM_ENABLE_CHUNKED_PREFILL": "true",
    # Reuses the KV of a shared prompt prefix across requests. A large win for
    # GSM8K-style prompts, which all end with the same instruction.
    "SM_VLLM_ENABLE_PREFIX_CACHING": "true",
    "SM_VLLM_KV_CACHE_DTYPE": "auto",
    "SM_VLLM_TENSOR_PARALLEL_SIZE": "1",
}

grpo_model = Model.create(
    model_name=model_name,
    primary_container=ContainerDefinition(
        image=image_uri,
        model_data_source=ModelDataSource(
            s3_data_source=S3ModelDataSource(
                s3_uri=model_s3_uri,
                s3_data_type="S3Prefix",
                compression_type="None",
            )
        ),
        environment=env,
    ),
    execution_role_arn=role,
)

pprint(grpo_model.model_name)

### Create Inference Component
An inference component is the unit that actually claims hardware on the endpoint, and its requirements describe what the *component needs*, not what the instance has.

That distinction caused a failure worth repeating. An earlier version of this lab asked for most of the host's memory, on the reasoning that the component should have the instance to itself, and SageMaker refused to place it: `There is not enough hardware resources on the instances for this endpoint to create a copy of the inference component`. Allocatable capacity sits below nameplate, because the platform holds memory and CPU back for its own agents. Reserving the host was also the wrong goal -- with one accelerator on the instance and the component claiming it, exclusivity is already guaranteed by `number_of_accelerator_devices_required`.

6 GiB is generous for what happens on the host here. The weights are read from the mounted prefix straight into VRAM, so host memory only carries the Python process, the mmap during load, and request buffers. None of that scales with the size of the card.

One more consequence of the instance pools above. When a variant lists pools rather than a single instance type, a component must supply a specification for *every* type in the list. A partial list is rejected outright with `Specifications must include an entry for every instance type in the InstancePools of the endpoint`, which is easy to act on. The singular `specification=` form is the dangerous one: the API accepts it, the call returns successfully, and then the component fails roughly a minute later with `Request to service failed. If failure persists after retry, contact customer support.` -- no container logs, because the container never starts. The list below is derived from `instance_pools` for that reason, so adding or reordering a pool cannot leave the two out of step.

In [ ]:
from sagemaker.core.resources import InferenceComponent
from sagemaker.core.shapes import (
    InferenceComponentComputeResourceRequirements,
    InferenceComponentRuntimeConfig,
    InferenceComponentSpecification,
)

# One specification per instance type in the endpoint's InstancePools, built from
# the same list so the two cannot drift apart.
#
# The plural form is required rather than stylistic. CreateInferenceComponent
# rejects a partial list outright -- "Specifications must include an entry for every
# instance type in the InstancePools of the endpoint" -- and the singular
# `specification=` form is worse: the API accepts it, returns success, and then the
# component fails about a minute later with "Request to service failed" and no
# container logs at all, because the container never starts.
#
# Requirements are identical across the three, since every candidate carries a
# single L40S. A fresh object per specification keeps them independent.
def _requirements():
    return InferenceComponentComputeResourceRequirements(
        min_memory_required_in_mb=6144,
        number_of_accelerator_devices_required=1,
        number_of_cpu_cores_required=4.0,
    )


inference_component = InferenceComponent.create(
    inference_component_name=ic_name,
    endpoint_name=endpoint_name,
    variant_name="AllTraffic",
    specifications=[
        InferenceComponentSpecification(
            instance_type=pool.instance_type,
            model_name=model_name,
            compute_resource_requirements=_requirements(),
        )
        for pool in instance_pools
    ],
    runtime_config=InferenceComponentRuntimeConfig(copy_count=1),
    region=region,
)

print(f"InferenceComponent created: {inference_component.inference_component_name}")
print("specifications:")
for pool in instance_pools:
    print(f"  {pool.instance_type}")
inference_component.wait_for_status("InService")
print(f"InferenceComponent {ic_name} is InService")

### Test endpoint
The endpoint speaks the OpenAI chat format. We send a GSM8K-shaped question with the same instruction Lab 1 appended to every training prompt, because that is the format the model was rewarded for producing -- the answer should arrive after a `####` marker.

Utility functions to read the response stream.

In [ ]:
import io


class LineIterator:
    """Yields complete lines from a SageMaker response stream.

    The stream delivers arbitrary byte chunks, not lines, so a chunk boundary can fall
    in the middle of a JSON payload. This buffers until a newline arrives.
    """

    def __init__(self, stream):
        self.byte_iterator = iter(stream)
        self.buffer = io.BytesIO()
        self.read_pos = 0

    def __iter__(self):
        return self

    def __next__(self):
        while True:
            self.buffer.seek(self.read_pos)
            line = self.buffer.readline()
            if line and line[-1] == ord("\n"):
                self.read_pos += len(line)
                return line[:-1]
            try:
                chunk = next(self.byte_iterator)
            except StopIteration:
                if self.read_pos < self.buffer.getbuffer().nbytes:
                    continue
                raise
            if "PayloadPart" not in chunk:
                continue
            self.buffer.seek(0, io.SEEK_END)
            self.buffer.write(chunk["PayloadPart"]["Bytes"])


def parse_streaming_chunk(line: str) -> str:
    """Pull the content delta out of one server-sent-event line."""
    if not line.startswith("data:"):
        return ""
    payload = line[len("data:") :].strip()
    if not payload or payload == "[DONE]":
        return ""
    choices = json.loads(payload).get("choices") or [{}]
    return choices[0].get("delta", {}).get("content") or ""

In [ ]:
INSTRUCTION = 'Let\'s think step by step and output the final answer after "####".'

question = (
    "Janet has 3 boxes with 12 pencils each. She gives away 7 pencils. "
    "How many pencils does she have left?"
)

request_body = {
    "messages": [{"role": "user", "content": f"{question} {INSTRUCTION}"}],
    # Matches max_response_length from Lab 2. The answer line comes last, so a
    # smaller budget truncates the completion before it is reached.
    "max_tokens": 1024,
    # Greedy, so the same question gives the same answer every time.
    "temperature": 0.0,
    "stream": True,
}

response = sagemaker_runtime.invoke_endpoint_with_response_stream(
    EndpointName=endpoint_name,
    InferenceComponentName=ic_name,
    Body=json.dumps(request_body),
    ContentType="application/json",
)

completion = ""
for line in LineIterator(response["Body"]):
    if not line:
        continue
    content = parse_streaming_chunk(line.decode("utf-8"))
    if content:
        completion += content
        print(content, end="", flush=True)

This proves the endpoint generates tokens, and that the trained model still produces the `####` format it was rewarded for. It is **not** an evaluation: one question is not a measurement, and nothing here checked the answer. Lab 4 does that.

***

### Delete resources
Run all four. The endpoint is the one that costs money while it exists; the others are free to hold but will block a later redeploy if left behind.

In [ ]:
from sagemaker.core.resources import InferenceComponent

# Delete inference component
InferenceComponent.get(inference_component_name=ic_name).delete()

In [ ]:
from sagemaker.core.resources import Endpoint

# Delete endpoint -- this is the one that stops the hourly charge
Endpoint.get(endpoint_name=endpoint_name).delete()

In [ ]:
from sagemaker.core.resources import Model

# Delete model
Model.get(model_name=model_name).delete()

In [ ]:
from sagemaker.core.resources import EndpointConfig

# Delete endpoint config
EndpointConfig.get(endpoint_config_name=endpoint_config_name).delete()